## **Install Dependencies**

In [1]:
# Jika environment belum ada library yang dibutuhkan
# !pip install torch transformers scikit-learn pandas numpy tqdm

## **Import Libraries**

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from tqdm import tqdm
from collections import defaultdict

## **Konfigurasi Global**

In [3]:
MODEL_NAME = "indobenchmark/indobert-base-p2"

LEVELS = [
    "level_1",
    "level_2",
    "level_3",
    "level_4",
    "level_5",
    "level_6"
]

MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 10
LR = 2e-5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

Device: cuda


## **Load Dataset**

In [ ]:
df = pd.read_csv("data training.csv")

df["text"] = (
    df["nama_produk"].fillna("") + " " +
    df["deskripsi"].fillna("")
).str.lower()

In [5]:
df.head()

,nama_produk,level_1,level_2,level_3,level_4,level_5,level_6,deskripsi,text
0,alat Pel Sumbu,barang,rumah tangga,kebersihan,alat pel,tongkat pel,NaN,alat pel lantai dengan sumbu,alat pel sumbu alat pel lantai dengan sumbu
1,Buku Guru : Model Penampang Udang,barang,buku teks,buku guru,semua jenjang,semua kelas,pendidikan khusus,buku panduan model penampang udang,buku guru : model penampang udang buku panduan...
2,AC AKARI Pendingin Ruangan 1/2 PK,barang,elektronik,alat pendingin ruangan,ac portable,NaN,NaN,pendingin ruangan kapasitas setengah PK,ac akari pendingin ruangan 1/2 pk pendingin ru...
3,Ac Daikin Split 0.75 PK Inverter FTKQ20 Gratis...,barang,elektronik,alat pendingin ruangan,ac portable,NaN,NaN,AC inverter hemat energi modern,ac daikin split 0.75 pk inverter ftkq20 gratis...
4,AC Sharp 1/2 Pk,barang,elektronik,alat pendingin ruangan,ac reflektor,NaN,NaN,pendingin ruangan kapasitas kecil rumah,ac sharp 1/2 pk pendingin ruangan kapasitas ke...


## **Label Encoding (Setiap Level)**

In [6]:
label_encoders = {}
num_classes = {}

for lvl in LEVELS:

    le = LabelEncoder()
    df[lvl] = le.fit_transform(df[lvl])

    label_encoders[lvl] = le
    num_classes[lvl] = len(le.classes_)

print(num_classes)

{'level_1': 4, 'level_2': 22, 'level_3': 183, 'level_4': 1250, 'level_5': 81, 'level_6': 347}


## **Membuat Hirarki (Mapping Parent dan Children)**

In [7]:
tree_maps = {}

for i in range(len(LEVELS) - 1):

    parent_lvl = LEVELS[i]
    child_lvl = LEVELS[i + 1]

    mapping = defaultdict(set)

    for _, row in df.iterrows():
        mapping[row[parent_lvl]].add(row[child_lvl])

    tree_maps[(parent_lvl, child_lvl)] = mapping

## **Train/ Validation Split**

In [8]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 16534
Validation size: 1838


## **Tokenizer**

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## **Menyiapkan Dataset**

In [10]:
class ProductDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        text = self.df.loc[idx, "text"]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        labels = {
            lvl: torch.tensor(self.df.loc[idx, lvl], dtype=torch.long)
            for lvl in LEVELS
        }

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": labels
        }

## **Train Val Loader**

In [11]:
train_loader = DataLoader(
    ProductDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    ProductDataset(val_df),
    batch_size=BATCH_SIZE
)

## **Membangun Model Hirarki**

In [12]:
class HierarchicalMaskedClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.encoder.config.hidden_size

        self.classifiers = nn.ModuleDict({
            lvl: nn.Linear(hidden_size, num_classes[lvl])
            for lvl in LEVELS
        })

    def forward(self, input_ids, attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]

        logits = {
            lvl: self.classifiers[lvl](cls_embedding)
            for lvl in LEVELS
        }

        return logits

## **Inisiasi Model**

In [13]:
model = HierarchicalMaskedClassifier().to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.parameters(),
    lr=LR
)

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

## **Model Train**

In [14]:
for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        labels = {
            lvl: batch["labels"][lvl].to(DEVICE)
            for lvl in LEVELS
        }

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)

        loss = 0
        for lvl in LEVELS:
            loss += criterion(outputs[lvl], labels[lvl])

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

100%|██████████| 1034/1034 [11:55<00:00,  1.44it/s]


Epoch 1 | Loss: 9.9738


100%|██████████| 1034/1034 [12:10<00:00,  1.42it/s]


Epoch 2 | Loss: 5.1023


100%|██████████| 1034/1034 [12:10<00:00,  1.42it/s]


...

Epoch 8 | Loss: 0.5160


100%|██████████| 1034/1034 [12:10<00:00,  1.42it/s]


Epoch 9 | Loss: 0.4171


100%|██████████| 1034/1034 [12:11<00:00,  1.41it/s]

Epoch 10 | Loss: 0.3335


## **Fungsi Inferensi**

In [15]:
def predict_text(text):

    model.eval()

    encoding = tokenizer(
        text.lower(),
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)

    predictions = {}
    parent_id = None

    for i, lvl in enumerate(LEVELS):

        logit = logits[lvl].clone()

        if i > 0 and parent_id is not None:

            parent_lvl = LEVELS[i-1]
            mapping = tree_maps[(parent_lvl, lvl)]

            valid_children = mapping.get(parent_id, [])

            if len(valid_children) == 0:
                break

            mask = torch.full_like(logit, -1e9)
            mask[:, list(valid_children)] = 0

            logit = logit + mask

        pred_id = torch.argmax(logit, dim=1).item()

        predictions[lvl] = label_encoders[lvl].inverse_transform([pred_id])[0]

        parent_id = pred_id

    return predictions

In [16]:
def predict_csv(input_csv, output_csv):

    new_df = pd.read_csv(input_csv)

    new_df["text"] = (
        new_df["nama_produk"].fillna("") +
        " " +
        new_df["deskripsi"].fillna("")
    ).str.lower()

    results = []

    for text in tqdm(new_df["text"]):
        pred = predict_text(text)
        results.append(pred)

    pred_df = pd.DataFrame(results)

    final_df = pd.concat(
        [new_df["nama_produk"], pred_df],
        axis=1
    )

    final_df.to_csv(output_csv, index=False)

    print("Prediction saved to:", output_csv)

In [17]:
predict_text(
    "laptop asus"
)

{'level_1': 'barang',
 'level_2': 'komputer & laptop',
 'level_3': 'laptop',
 'level_4': 'notebook',
 'level_5': nan,
 'level_6': nan}

In [18]:
predict_csv(
    "scrapping_uji coba.csv",
    "hasil_prediksi.csv"
)

100%|██████████| 2020/2020 [00:40<00:00, 49.28it/s]

Prediction saved to: hasil_prediksi.csv
